# 05 — Launch vLLM as an OpenAI-Compatible Server

Goal: start vLLM as an API server and call it using the OpenAI Python client. Run the server in a terminal before executing the client cells.

Start the server from the repo root:

```bash
./scripts/run_vllm_server.sh
```

Or manually:

```bash
vllm serve TinyLlama/TinyLlama-1.1B-Chat-v1.0 --host 0.0.0.0 --port 8000 --dtype auto --max-model-len 2048 --max-num-seqs 16 --gpu-memory-utilization 0.85
```

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

from openai import OpenAI
from vllm_lab.utils import load_config

config = load_config(repo_root / 'configs' / 'lab_config.yaml')
base_url = config['vllm_server']['base_url']
model = config['model']['default_name']
client = OpenAI(base_url=base_url, api_key='EMPTY')
print(base_url, model)

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': 'Explain PagedAttention in three bullet points.'}],
    temperature=0.0,
    max_tokens=120,
)
print(response.choices[0].message.content)
print(response.usage)

If this fails, check whether the vLLM server is actually running and whether the model name in the request exactly matches the served model.

In [ ]:
# Sequential API benchmark: useful as a baseline before concurrent testing.
import pandas as pd
from vllm_lab.benchmark import benchmark_openai_sync

rows = benchmark_openai_sync(base_url=base_url, model=model, prompts=config['benchmark']['prompts'], max_tokens=128)
df = pd.DataFrame(rows)
df

In [ ]:
out = repo_root / 'results' / 'vllm_openai_sync.csv'
df.to_csv(out, index=False)
print('wrote', out)